In [ ]:
pip install delta-spark==2.4.0

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType

warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

spark = (
    SparkSession.builder
    .appName("DataCo ETL")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
    .getOrCreate()
)

# Período analisado: de 01/01/2017 a 30/09/2017
# (depois de 30/09/2017 as encomendas têm ~1 item e metade do valor, o que sugere registos incompletos)
DATA_MIN = "2017-01-01"
DATA_MAX = "2017-09-30"

In [ ]:
schema_dataco = StructType([
    StructField("type", StringType(), True),
    StructField("days_for_shipping_real", IntegerType(), True),
    StructField("days_for_shipment_scheduled", IntegerType(), True),
    StructField("benefit_per_order", DoubleType(), True),
    StructField("sales_per_customer", DoubleType(), True),
    StructField("delivery_status", StringType(), True),
    StructField("late_delivery_risk", IntegerType(), True),
    StructField("category_id", IntegerType(), True),
    StructField("category_name", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_country", StringType(), True),
    StructField("customer_email", StringType(), True),
    StructField("customer_fname", StringType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("customer_lname", StringType(), True),
    StructField("customer_password", StringType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("customer_state", StringType(), True),
    StructField("customer_street", StringType(), True),
    StructField("customer_zipcode", DoubleType(), True),
    StructField("department_id", IntegerType(), True),
    StructField("department_name", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("market", StringType(), True),
    StructField("order_city", StringType(), True),
    StructField("order_country", StringType(), True),
    StructField("order_customer_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("order_id", IntegerType(), True),
    StructField("order_item_cardprod_id", IntegerType(), True),
    StructField("order_item_discount", DoubleType(), True),
    StructField("order_item_discount_rate", DoubleType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("order_item_product_price", DoubleType(), True),
    StructField("order_item_profit_ratio", DoubleType(), True),
    StructField("order_item_quantity", IntegerType(), True),
    StructField("sales", DoubleType(), True),
    StructField("order_item_total", DoubleType(), True),
    StructField("order_profit_per_order", DoubleType(), True),
    StructField("order_region", StringType(), True),
    StructField("order_state", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_zipcode", DoubleType(), True),
    StructField("product_card_id", IntegerType(), True),
    StructField("product_category_id", IntegerType(), True),
    StructField("product_description", DoubleType(), True),
    StructField("product_image", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("product_price", DoubleType(), True),
    StructField("product_status", IntegerType(), True),
    StructField("shipping_date", StringType(), True),
    StructField("shipping_mode", StringType(), True),
])

csv_path = "hdfs://hdfs-nn:9000/projeto/bronze/DataCoSupplyChainDataset.csv"

df = spark.read.csv(
    csv_path,
    header=True,
    schema=schema_dataco,
    encoding="ISO-8859-1",
    quote='"',
    escape='"'
)

print(f"Total de registos lidos: {df.count()}")      # esperado: 180519
df.show(5, truncate=30)

In [ ]:
print("\n=== Início da Limpeza e Transformação ===")

# 1. Eliminar colunas (duplicadas, pessoais, vazias, constantes, deriváveis ou sem utilidade analítica)
colunas_eliminar = [
    "sales_per_customer", "benefit_per_order", "order_item_cardprod_id",
    "customer_fname", "customer_lname", "customer_street",
    "product_description", "product_status",
    "customer_email", "customer_password", "order_zipcode",
    "order_customer_id", "product_category_id", "product_image", "product_price",
    "latitude", "longitude", "order_item_discount_rate", "order_item_profit_ratio",
    "customer_city", "customer_state", "customer_zipcode", "customer_country",
    "delivery_status",
]
df_clean = df.drop(*colunas_eliminar)
print(f"TRATAMENTO: Eliminadas {len(colunas_eliminar)} colunas. Restam {len(df_clean.columns)}.")   # esperado: 29 restantes

# 2. Converter datas (texto -> timestamp)
FMT = "M/d/yyyy H:mm"
df_clean = (df_clean
    .withColumn("order_date", F.to_timestamp("order_date", FMT))
    .withColumn("shipping_date", F.to_timestamp("shipping_date", FMT)))

total_antes = df_clean.count()
df_clean = df_clean.filter(F.col("order_date").isNotNull() & F.col("shipping_date").isNotNull())
print(f"TRATAMENTO (Filtro datas): Removidos {total_antes - df_clean.count()} registos por data inválida.")   # esperado: 0

# 3. Limpar texto: remover espaços no início/fim e duplos, e strings vazias passam a nulo
string_cols = [c.name for c in df_clean.schema.fields if c.dataType == StringType()]
for column in string_cols:
    df_clean = df_clean.withColumn(column, F.regexp_replace(F.trim(F.col(column)), r"\s+", " "))
    df_clean = df_clean.withColumn(column, F.when(F.col(column) == "", None).otherwise(F.col(column)))
print("TRATAMENTO: Removidos espaços a mais e convertidas strings vazias para nulo.")

# 4. Categorias: cada category_name fica com um único category_id (o mais frequente)
w = Window.partitionBy("category_name").orderBy(F.col("n").desc(), F.col("category_id"))
cat_ok = (df_clean.groupBy("category_name", "category_id").agg(F.count("*").alias("n"))
            .withColumn("rn", F.row_number().over(w))
            .filter("rn = 1")
            .select("category_name", F.col("category_id").alias("category_id_ok")))

df_clean = (df_clean.join(cat_ok, "category_name", "left")
            .drop("category_id")
            .withColumnRenamed("category_id_ok", "category_id"))
print("TRATAMENTO: Uniformizado category_id por category_name.")

# 5. Recalcular dias reais (dias de calendário) e o indicador de atraso (dias reais > dias agendados)
dias_reais = F.datediff(F.to_date("shipping_date"), F.to_date("order_date"))
print("Diferenças face aos dias reais originais:",
      df_clean.filter(dias_reais != F.col("days_for_shipping_real")).count())      # esperado: 0

df_clean = (df_clean
    .withColumn("days_for_shipping_real", dias_reais.cast("int"))
    .withColumn("late_delivery_risk",
                F.when(F.col("days_for_shipping_real") > F.col("days_for_shipment_scheduled"), 1).otherwise(0)))
print("TRATAMENTO: Recalculados os dias reais e o indicador de atraso.")

# 6. Excluir encomendas canceladas e de fraude
excl = F.col("order_status").isin("CANCELED", "SUSPECTED_FRAUD")
print(f"TRATAMENTO (Filtro order_status): Removidas {df_clean.filter(excl).count()} linhas "
      f"({df_clean.filter(excl).select('order_id').distinct().count()} encomendas).")   # esperado: 7754 linhas e 2855 encomendas
df_clean = df_clean.filter(~excl)

# 7. Período analisado
total_antes = df_clean.count()
df_clean = df_clean.filter((F.to_date("order_date") >= DATA_MIN) & (F.to_date("order_date") <= DATA_MAX))
print(f"TRATAMENTO (Filtro período): Removidos {total_antes - df_clean.count()} itens fora de {DATA_MIN} a {DATA_MAX}.")   # esperado: 128073

# 8. Ano da encomenda (partição) e data da encomenda (chave de ligação por data)
df_clean = (df_clean
    .withColumn("order_year", F.year("order_date"))
    .withColumn("data_encomenda", F.to_date("order_date")))

In [ ]:
print("\n=== Validações ===")

n_itens = df_clean.count()
assert n_itens == df_clean.select("order_item_id").distinct().count(), "order_item_id repetido"

colunas_kpi = ["order_id", "order_item_id", "sales", "order_profit_per_order", "market", "category_name",
               "shipping_mode", "order_status", "days_for_shipping_real", "days_for_shipment_scheduled",
               "late_delivery_risk"]
nulos = df_clean.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas_kpi]).first().asDict()
assert sum(nulos.values()) == 0, nulos

# atributos da encomenda iguais em todas as linhas da mesma encomenda
col_enc = ["order_status", "market", "order_region", "order_country", "shipping_mode", "late_delivery_risk",
           "days_for_shipping_real", "days_for_shipment_scheduled", "order_date", "shipping_date"]
variam = (df_clean.groupBy("order_id")
            .agg(*[F.countDistinct(c).alias(c) for c in col_enc])
            .agg(*[F.max(c).alias(c) for c in col_enc]).first().asDict())
assert all(v == 1 for v in variam.values()), variam

print(f"Todas as validações passaram. Itens: {n_itens} | Encomendas: {df_clean.select('order_id').distinct().count()}")
# esperado: 44692 itens e 14904 encomendas

In [ ]:
def guardar_silver(df_tabela, nome):
    silver_path = f"hdfs://hdfs-nn:9000/projeto/silver/{nome}"
    print(f"\n=== A guardar dados limpos em {silver_path} ===")

    df_tabela.write \
        .mode("overwrite") \
        .format("parquet") \
        .partitionBy("order_year") \
        .save(silver_path)

    colunas = ",\n        ".join(f"`{c}` {t.upper()}" for c, t in df_tabela.dtypes if c != "order_year")

    spark.sql(f"DROP TABLE IF EXISTS silver.{nome}")
    spark.sql(f"""
        CREATE EXTERNAL TABLE silver.{nome} (
        {colunas}
        )
        PARTITIONED BY (order_year INT)
        STORED AS PARQUET
        LOCATION '{silver_path}'
    """)
    spark.catalog.recoverPartitions(f"silver.{nome}")
    print(f"Tabela Parquet particionada criada com sucesso em silver.{nome}")


spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{warehouse_location}/silver.db'")

guardar_silver(df_clean, "dataco_itens")

In [ ]:
print("\n=== Itens e encomendas por mês ===")
spark.sql("""
    SELECT date_format(order_date, 'yyyy-MM') AS mes,
           COUNT(*) AS total_itens, COUNT(DISTINCT order_id) AS total_encomendas
    FROM silver.dataco_itens
    GROUP BY 1
    ORDER BY 1
""").show(20)
# esperado: de 2017-01 a 2017-09, entre 4675 e 5145 itens e entre 1538 e 1710 encomendas por mês

print("\n=== Encomendas e taxa de atraso por modo de envio ===")
spark.sql("""
    SELECT shipping_mode,
           COUNT(DISTINCT order_id) AS encomendas,
           ROUND(COUNT(DISTINCT CASE WHEN late_delivery_risk = 1 THEN order_id END)
                 / COUNT(DISTINCT order_id) * 100, 1) AS pct_atrasadas
    FROM silver.dataco_itens
    GROUP BY shipping_mode
    ORDER BY shipping_mode
""").show()
# esperado: First Class 2234 (100.0) | Same Day 813 (52.3) | Second Class 2888 (79.5) | Standard Class 8969 (39.3)

print("\n=== Encomendas, vendas, lucro e margem por mercado ===")
spark.sql("""
    SELECT market, COUNT(DISTINCT order_id) AS encomendas,
           ROUND(SUM(sales), 0) AS vendas, ROUND(SUM(order_profit_per_order), 0) AS lucro,
           ROUND(SUM(order_profit_per_order) / SUM(sales) * 100, 1) AS margem_pct
    FROM silver.dataco_itens
    GROUP BY market
    ORDER BY market
""").show()
# esperado (encomendas): LATAM 8211 | Europe 5910 | Africa 395 | Pacific Asia 350 | USCA 38

spark.stop()

In [ ]:
from pyspark.sql import SparkSession

# ATENÇÃO: esta célula tem de correr num kernel novo (Kernel > Restart Kernel), para o Spark carregar o Delta.
# Se a conversão der PARSE_SYNTAX_ERROR em 'CONVERT', reinicia o kernel e corre só as 3 últimas células.

# --- 1. CONFIGURAÇÃO DA SPARKSESSION (com Delta) ---
# Esta célula precisa da sua própria SparkSession, que carrega as bibliotecas do Delta.
print("A iniciar a SparkSession com o pacote Delta Lake...")

warehouse_location = "hdfs://hdfs-nn:9000/warehouse"
hive_uris = "thrift://hive-metastore:9083"

spark = (
    SparkSession.builder
    .appName("Delta Converter (DataCo)")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", hive_uris)

    # --- Configuração essencial do Delta ---
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    .enableHiveSupport()
    .getOrCreate()
)

print("Sessão Spark com Delta pronta.")

In [ ]:
# --- 2. O COMANDO DE CONVERSÃO ---
# Não re-escreve os dados Parquet: apenas acrescenta o '_delta_log' para "adotar" os ficheiros existentes.
# Como as tabelas estão particionadas, é obrigatório indicar a coluna de partição.
tabelas = {
    "silver.dataco_itens": "order_year"
}

for tabela, particao in tabelas.items():
    print(f"A converter a tabela Parquet '{tabela}' para Delta Lake...")
    try:
        spark.sql(f"""
            CONVERT TO DELTA {tabela}
            PARTITIONED BY ({particao} INT)
        """)
        print(" Conversão concluída com sucesso!")
    except Exception as e:
        print(" Ocorreu um erro durante a conversão.")
        print("Causas prováveis: 1) A tabela não existe. 2) A tabela já é Delta.")
        print(e)

In [ ]:
# --- 3. VERIFICAÇÃO ---
for tabela in tabelas:
    spark.sql(f"DESCRIBE DETAIL {tabela}").select("format", "location", "numFiles", "partitionColumns").show(truncate=False)

spark.sql("SELECT COUNT(*) AS itens, COUNT(DISTINCT order_id) AS encomendas FROM silver.dataco_itens").show()   # esperado: 44692 e 14904

# (Opcional) Podes parar esta sessão
# spark.stop()